# Installation de HypergraphPercol

In [ ]:
# @title Configuration du Backend\n# Choix du backend géométrique : 'geogram' (Rapide, Recommandé) ou 'cgal' (Historique, Exact)\nBACKEND = 'geogram'  # @param ['geogram', 'cgal']\nprint(f'Backend sélectionné : {BACKEND}')\n

In [ ]:
import os\nimport subprocess\n\ndef run_sh(cmd):\n    print(f'$ {cmd}')\n    subprocess.check_call(cmd, shell=True)\n\nprint('Installation des outils de build...')\nrun_sh('apt-get update -qq')\n# libeigen3-dev est requis pour HGP (kernels_geogram.hpp)\nrun_sh('apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev')\n\nif BACKEND == 'cgal':\n    print('Installation des dépendances CGAL/TBB...')\n    run_sh('apt-get install -y -qq libcgal-dev libtbb-dev libtbbmalloc2 libgmp-dev libmpfr-dev')\nelif BACKEND == 'geogram':\n    print('Backend Geogram: Pas de dépendances système supplémentaires (Build headless).')\n    # Si le build échoue à cause de X11 manquant, décommentez la ligne suivante :\n    # run_sh('apt-get install -y -qq libx11-dev libxrandr-dev libxcursor-dev libxi-dev libxinerama-dev libgl1-mesa-dev libglu1-mesa-dev')\n\n

In [ ]:
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi\n

In [ ]:
%%bash\nset -euo pipefail\nWORKDIR="${HGP_WORKDIR:-/content}"\nmkdir -p "${WORKDIR}"\ncd "${WORKDIR}"\n\n# HGP-clusterer\nif [ -d HGP-clusterer ]; then\n    echo 'Mise à jour de HGP-clusterer (Force Reset)...'\n    cd HGP-clusterer\n    git fetch origin\n    git reset --hard origin/main\n    cd ..\nelse\n    git clone https://github.com/Ludwig-H/HGP-clusterer.git\nfi\n\n# Cyminiball\nif [ -d cyminiball ]; then\n    echo 'Mise à jour de cyminiball...'\n    cd cyminiball\n    git pull --ff-only\n    cd ..\nelse\n    git clone https://github.com/Ludwig-H/cyminiball.git\nfi\n

In [ ]:
%%bash\nset -euo pipefail\nWORKDIR="${HGP_WORKDIR:-/content}"\nmkdir -p "${WORKDIR}/wheels"\ncd "${WORKDIR}/cyminiball"\npython3 -m pip wheel --no-build-isolation --no-deps --wheel-dir="${WORKDIR}/wheels" .\npython3 -m pip install --force-reinstall --no-deps --no-index --find-links="${WORKDIR}/wheels" cyminiball\n

In [ ]:
import os\nimport sys\nfrom pathlib import Path\n\nWORKDIR = os.environ.get('HGP_WORKDIR', '/content')\nos.chdir(WORKDIR)\n\nif BACKEND == 'geogram':\n    if not os.path.exists('geogram'):\n        print('Clonage de Geogram...')\n        get_ipython().system('git clone --recursive https://github.com/BrunoLevy/geogram.git')\n    \n    print('Compilation de Geogram (Headless)...')\n    # Utilisation directe de CMake pour désactiver les graphismes (évite les dépendances X11)\n    get_ipython().system('cmake -S geogram -B geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF')\n    get_ipython().system('cmake --build geogram/build --config Release --parallel 4')\n    get_ipython().system('cmake --install geogram/build --prefix /usr/local')\n    \n    os.environ['GEOGRAM_INSTALL_PREFIX'] = '/usr/local'\n    print('Geogram installé.')\n\nelif BACKEND == 'cgal':\n    print('Configuration des dépendances CGAL locales...')\n    setup_script = f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"\n    get_ipython().system(f'python3 {setup_script}')\n    \n    cgal_dir = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"\n    projects = [\n        'EdgesCGALDelaunay2D', 'EdgesCGALDelaunay3D', 'EdgesCGALDelaunayND',\n        'EdgesCGALWeightedDelaunay2D', 'EdgesCGALWeightedDelaunay3D', 'EdgesCGALWeightedDelaunayND'\n    ]\n    for proj in projects:\n        p_path = f"{cgal_dir}/{proj}"\n        get_ipython().system(f'cmake -S {p_path} -B {p_path}/build -DCMAKE_BUILD_TYPE=Release')\n        get_ipython().system(f'cmake --build {p_path}/build --config Release')\n        get_ipython().system(f'cmake --install {p_path}/build --prefix {WORKDIR}/HGP-clusterer')\n

In [ ]:
%%bash\nset -euo pipefail\nWORKDIR="${HGP_WORKDIR:-/content}"\ncd "${WORKDIR}/HGP-clusterer"\n\necho "Installation de HGP-clusterer (Backend détecté par CMake)..."\npython3 -m pip install -v --no-deps .\n

In [8]:
import os

workdir = os.environ.get("HGP_WORKDIR", "/content")
repo_root = os.path.join(workdir, "HGP-clusterer")
os.environ["CGALDELAUNAY_ROOT"] = os.path.join(repo_root, "CGALDelaunay")

from hgp_clusterer import HypergraphPercol

In [9]:
# import hgp_clusterer

# # List all attributes (functions, classes, variables) in hgp_clusterer
# print("Functions and attributes in hgp_clusterer:")
# for item in dir(hgp_clusterer):
#     # if not item.startswith('_'): # Exclude private/special methods
#     print(item)


Functions and attributes in hgp_clusterer:
HypergraphPercol


In [42]:
# Cellule 2 : imports, paramètres globaux, wrappers

import numpy as np
import pandas as pd

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from scipy.optimize import linear_sum_assignment

import hdbscan
import plotly.express as px

from IPython.display import display

import plotly.io as pio
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot

# Active le mode offline pour Colab
init_notebook_mode(connected=True)


# Pour que Plotly s'affiche correctement dans Colab
pio.renderers.default = "colab"

# -------------------------------------------------------------------
# Paramètres globaux
# -------------------------------------------------------------------
NOISE_LABEL = -1   # label pour le bruit dans y_true et y_pred

# Paramètres pour HypergraphPercol & HDBSCAN
K = 5
min_cluster_size = 50
min_samples = K + 1         # tu peux mettre None si tu veux tester autre chose
method = 'eom'
splitting = None
weight_face = "lambda"      # "lambda" ∝ 1/r ; "uniform" ∝ 1 ; "unique" 1 sur la face min r
label_all_points = False
return_multi_clusters = False
complex_chosen = "orderk_delaunay"
expZ = 3
cgal_root = "/content/HGP-clusterer/CGALDelaunay"
verbeux = True

# -------------------------------------------------------------------
# Import / wrapper HypergraphPercol
# -------------------------------------------------------------------

try:
    # Adapte si le module est ailleurs
    from hgp_clusterer import HypergraphPercol
except ImportError:
    HypergraphPercol = None
    print("⚠️ HypergraphPercol introuvable. Adapte l'import ci-dessus.")


def run_hypergraphpercol(X):
    """
    Lance HypergraphPercol avec les paramètres fournis.
    """
    if HypergraphPercol is None:
        raise ImportError("HypergraphPercol n'est pas importé. Corrige l'import dans la cellule 2.")

    labels = HypergraphPercol(
        M=X,
        K=K,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        method=method,
        splitting=splitting,
        weight_face=weight_face,
        label_all_points=label_all_points,
        return_multi_clusters=return_multi_clusters,
        complex_chosen=complex_chosen,
        expZ=expZ,
        cgal_root=cgal_root,
        verbeux=verbeux,
    )
    labels = np.asarray(labels)
    # On a besoin d'un vecteur 1D
    return labels.reshape(-1).astype(int)


def run_hdbscan(X):
    """
    Lance HDBSCAN avec tes paramètres.
    """
    ms = min_samples if min_samples is not None else min_cluster_size

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=ms,
        cluster_selection_method=method,
        allow_single_cluster=True,
    )
    labels = clusterer.fit_predict(X)
    labels = np.asarray(labels)
    return labels.reshape(-1).astype(int)


In [43]:
# Cellule 3 : génération de nuages 2D et 3D (bruit en -1)

def generate_2d_datasets(n_samples=3000, random_state=0):
    rng = np.random.RandomState(random_state)
    datasets = []

    # 1) Blobs avec variances très différentes (pas de bruit)
    X1, y1 = make_blobs(
        n_samples=n_samples,
        centers=4,
        cluster_std=[0.2, 1.0, 2.5, 0.5],
        random_state=random_state
    )
    datasets.append({"name": "blobs_varies_2d", "X": X1, "y_true": y1})

    # 2) Deux lunes + bruit gaussien
    X2_core, y2_core = make_moons(n_samples=n_samples // 2, noise=0.07, random_state=random_state)
    noise = rng.normal(scale=0.5, size=(n_samples // 2, 2))

    X2 = np.vstack([X2_core, noise])
    y2 = np.concatenate([
        y2_core,                                  # 0, 1
        np.full(noise.shape[0], NOISE_LABEL)      # bruit = -1
    ])
    datasets.append({"name": "moons_plus_bruit_2d", "X": X2, "y_true": y2})

    # 3) Anneaux concentriques + outliers
    X3_core, y3_core = make_circles(
        n_samples=n_samples,
        factor=0.4,
        noise=0.06,
        random_state=random_state
    )
    extra = rng.uniform(low=-3.0, high=3.0, size=(n_samples // 10, 2))

    X3 = np.vstack([X3_core, extra])
    y3 = np.concatenate([
        y3_core,                                  # 0, 1
        np.full(extra.shape[0], NOISE_LABEL)      # bruit = -1
    ])
    datasets.append({"name": "cercles_plus_outliers_2d", "X": X3, "y_true": y3})

    return datasets


def generate_3d_datasets(n_samples=6000, random_state=0):
    rng = np.random.RandomState(random_state)
    datasets = []

    # 1) Blobs anisotropes 3D (pas de bruit)
    X1, y1 = make_blobs(
        n_samples=n_samples,
        centers=4,
        cluster_std=[0.3, 1.5, 0.5, 2.0],
        n_features=3,
        random_state=random_state
    )
    transformation = np.array([
        [0.6, -0.6, 0.3],
        [0.4,  0.8, -0.5],
        [0.3,  0.2,  0.7],
    ])
    X1 = X1.dot(transformation)
    datasets.append({"name": "blobs_anisotropes_3d", "X": X1, "y_true": y1})

    # 2) Deux spirales 3D entremêlées + bruit
    t = np.linspace(0, 4 * np.pi, n_samples // 3)
    x1 = np.cos(t)
    y1s = np.sin(t)
    z1 = t / (4 * np.pi)

    x2 = np.cos(t + np.pi)
    y2s = np.sin(t + np.pi)
    z2 = t / (4 * np.pi)

    spiral1 = np.vstack([x1, y1s, z1]).T
    spiral2 = np.vstack([x2, y2s, z2]).T

    spiral1 += rng.normal(scale=0.05, size=spiral1.shape)
    spiral2 += rng.normal(scale=0.05, size=spiral2.shape)

    noise = rng.uniform(low=-2.0, high=2.0, size=(n_samples - 2 * spiral1.shape[0], 3))

    X2 = np.vstack([spiral1, spiral2, noise])
    y2 = np.concatenate([
        np.zeros(spiral1.shape[0], dtype=int),
        np.ones(spiral2.shape[0], dtype=int),
        np.full(noise.shape[0], NOISE_LABEL, int)
    ])
    datasets.append({"name": "spirales_plus_bruit_3d", "X": X2, "y_true": y2})

    return datasets


# Génération et quick check
datasets_2d = generate_2d_datasets()
datasets_3d = generate_3d_datasets()

for d in datasets_2d + datasets_3d:
    k_true = len(np.unique(d["y_true"][d["y_true"] != NOISE_LABEL]))
    print(d["name"], d["X"].shape, "k_true (sans bruit) =", k_true)


blobs_varies_2d (3000, 2) k_true (sans bruit) = 4
moons_plus_bruit_2d (3000, 2) k_true (sans bruit) = 2
cercles_plus_outliers_2d (3300, 2) k_true (sans bruit) = 2
blobs_anisotropes_3d (6000, 3) k_true (sans bruit) = 4
spirales_plus_bruit_3d (6000, 3) k_true (sans bruit) = 2


In [44]:
# Cellule 4 : appariement hongrois, matrices de confusion, métriques

def make_confusion_df(y_true, y_pred, labels_true=None, labels_pred=None):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    if labels_true is None:
        labels_true = sorted(np.unique(y_true))
    if labels_pred is None:
        labels_pred = sorted(np.unique(y_pred))

    idx_true = {lab: i for i, lab in enumerate(labels_true)}
    idx_pred = {lab: j for j, lab in enumerate(labels_pred)}

    cm = np.zeros((len(labels_true), len(labels_pred)), dtype=int)

    for t, p in zip(y_true, y_pred):
        if t in idx_true and p in idx_pred:
            cm[idx_true[t], idx_pred[p]] += 1

    index = [f"true_{lab}" for lab in labels_true]
    columns = [f"pred_{lab}" for lab in labels_pred]
    return pd.DataFrame(cm, index=index, columns=columns)


def hungarian_relabel(y_true, y_pred, noise_label=NOISE_LABEL):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    assert y_true.shape == y_pred.shape, "y_true et y_pred doivent avoir la même longueur."

    mask = (y_true != noise_label) & (y_pred != noise_label)
    y_true_valid = y_true[mask]
    y_pred_valid = y_pred[mask]

    true_labels = sorted(np.unique(y_true_valid))
    pred_labels = sorted(np.unique(y_pred_valid))

    if len(true_labels) == 0 or len(pred_labels) == 0:
        mapping = {}
        y_pred_aligned = np.copy(y_pred)
        confusion_after = make_confusion_df(y_true, y_pred_aligned)
        return y_pred_aligned, mapping, confusion_after

    cm = np.zeros((len(true_labels), len(pred_labels)), dtype=int)
    idx_true = {lab: i for i, lab in enumerate(true_labels)}
    idx_pred = {lab: j for j, lab in enumerate(pred_labels)}

    for t, p in zip(y_true_valid, y_pred_valid):
        cm[idx_true[t], idx_pred[p]] += 1

    cost_matrix = cm.max() - cm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    mapping = {pred_labels[c]: true_labels[r] for r, c in zip(row_ind, col_ind)}

    y_pred_aligned = np.empty_like(y_pred)
    for i, lbl in enumerate(y_pred):
        if lbl == noise_label:
            y_pred_aligned[i] = noise_label
        else:
            y_pred_aligned[i] = mapping.get(lbl, lbl)

    labels_true_all = sorted(np.unique(y_true))
    labels_pred_all = sorted(np.unique(y_pred_aligned))
    confusion_after = make_confusion_df(
        y_true, y_pred_aligned,
        labels_true=labels_true_all,
        labels_pred=labels_pred_all
    )

    return y_pred_aligned, mapping, confusion_after


def compute_metrics(y_true, y_pred_aligned, noise_label=NOISE_LABEL):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred_aligned = np.asarray(y_pred_aligned).reshape(-1)

    mask = (y_true != noise_label) & (y_pred_aligned != noise_label)
    if mask.sum() == 0:
        return {"ARI": np.nan, "NMI": np.nan}

    ari = adjusted_rand_score(y_true[mask], y_pred_aligned[mask])
    nmi = normalized_mutual_info_score(y_true[mask], y_pred_aligned[mask])

    return {"ARI": ari, "NMI": nmi}


In [45]:
# Cellule 5 : évaluation HypergraphPercol vs HDBSCAN + tableaux hongrois

results = []

labels_hgp_aligned = {}
labels_hdb_aligned = {}

all_datasets = datasets_2d + datasets_3d

for d in all_datasets:
    name = d["name"]
    X = d["X"]
    y_true = d["y_true"]
    k_true = len(np.unique(y_true[y_true != NOISE_LABEL]))

    print("\n" + "=" * 70)
    print(f"Dataset : {name} | n = {len(X)} | k_true (sans bruit) = {k_true}")
    print("=" * 70)

    row = {"dataset": name, "k_true": k_true}

    # ----------------------- HypergraphPercol -----------------------
    try:
        y_hgp = run_hypergraphpercol(X)
        y_hgp_aligned, mapping_hgp, cm_hgp = hungarian_relabel(y_true, y_hgp, noise_label=NOISE_LABEL)
        metrics_hgp = compute_metrics(y_true, y_hgp_aligned, noise_label=NOISE_LABEL)

        labels_hgp_aligned[name] = y_hgp_aligned

        print("\nHypergraphPercol : mapping (pred_label -> true_label)")
        mapping_df_hgp = pd.DataFrame(
            [{"pred_label": p, "true_label": t} for p, t in mapping_hgp.items()]
        )
        display(mapping_df_hgp)

        print("HypergraphPercol : matrice de confusion (après relabel)")
        display(cm_hgp)

        row["ARI_HGP"] = metrics_hgp["ARI"]
        row["NMI_HGP"] = metrics_hgp["NMI"]
    except Exception as e:
        print(f"Erreur HypergraphPercol sur {name} : {e}")
        labels_hgp_aligned[name] = np.full_like(y_true, NOISE_LABEL)
        row["ARI_HGP"] = np.nan
        row["NMI_HGP"] = np.nan

    # --------------------------- HDBSCAN ---------------------------
    try:
        y_hdb = run_hdbscan(X)
        y_hdb_aligned, mapping_hdb, cm_hdb = hungarian_relabel(y_true, y_hdb, noise_label=NOISE_LABEL)
        metrics_hdb = compute_metrics(y_true, y_hdb_aligned, noise_label=NOISE_LABEL)

        labels_hdb_aligned[name] = y_hdb_aligned

        print("\nHDBSCAN : mapping (pred_label -> true_label)")
        mapping_df_hdb = pd.DataFrame(
            [{"pred_label": p, "true_label": t} for p, t in mapping_hdb.items()]
        )
        display(mapping_df_hdb)

        print("HDBSCAN : matrice de confusion (après relabel)")
        display(cm_hdb)

        row["ARI_HDB"] = metrics_hdb["ARI"]
        row["NMI_HDB"] = metrics_hdb["NMI"]
    except Exception as e:
        print(f"Erreur HDBSCAN sur {name} : {e}")
        labels_hdb_aligned[name] = np.full_like(y_true, NOISE_LABEL)
        row["ARI_HDB"] = np.nan
        row["NMI_HDB"] = np.nan

    results.append(row)

print("\n===== Récapitulatif ARI / NMI =====")
results_df = pd.DataFrame(results)
display(results_df)



Dataset : blobs_varies_2d | n = 3000 | k_true (sans bruit) = 4
orderk_delaunay k = 1
Computed weighted barycentres 8989
orderk_delaunay k = 2
Computed weighted barycentres 25643
orderk_delaunay k = 3
Computed weighted barycentres 39665
orderk_delaunay k = 4
Computed weighted barycentres 53617
orderk_delaunay k = 5
Simplexes sans filtration : 67268
N_CPU_dispo utilisés :  7
5-simplices=67268
Faces uniques: 125985 (compression 403608→125985)
W_nodes calculé.
Arêtes uniques (U<V): 336340
Arêtes triées.
Kruskal appliqué. Nombre de composantes connexes : 1
[GetClusters] method=eom -> 4 clusters

HypergraphPercol : mapping (pred_label -> true_label)


,pred_label,true_label
0,0,0
1,2,1
2,3,2
3,1,3


HypergraphPercol : matrice de confusion (après relabel)


,pred_-1,pred_0,pred_1,pred_2,pred_3
true_0,0,750,0,0,0
true_1,32,0,716,2,0
true_2,431,36,119,101,63
true_3,0,0,0,0,750



HDBSCAN : mapping (pred_label -> true_label)


...

,pred_label,true_label
0,0,0


HDBSCAN : matrice de confusion (après relabel)


,pred_-1,pred_0,pred_1
true_-1,1287,653,60
true_0,0,2000,0
true_1,0,2000,0



===== Récapitulatif ARI / NMI =====


,dataset,k_true,ARI_HGP,NMI_HGP,ARI_HDB,NMI_HDB
0,blobs_varies_2d,4,0.851249,0.823448,0.789334,0.810176
1,moons_plus_bruit_2d,2,0.470822,0.487370,0.000000,0.000000
2,cercles_plus_outliers_2d,2,1.000000,1.000000,1.000000,1.000000
3,blobs_anisotropes_3d,4,0.979422,0.966628,0.979992,0.967124
4,spirales_plus_bruit_3d,2,0.000000,0.000000,0.000000,0.000000


In [46]:
# Cellule 6 : visualisations 2D avec Plotly Express

def plot_2d_results(X, y_true, y_hgp, y_hdb, title_prefix=""):
    X = np.asarray(X, dtype=float)
    y_true = np.asarray(y_true).reshape(-1)
    y_hgp = np.asarray(y_hgp).reshape(-1)
    y_hdb = np.asarray(y_hdb).reshape(-1)

    assert len(X) == len(y_true) == len(y_hgp) == len(y_hdb)

    df = pd.DataFrame({
        "x": X[:, 0],
        "y": X[:, 1],
        "true": y_true,
        "hypergraphpercol": y_hgp,
        "hdbscan": y_hdb,
    })

    df["true_str"] = df["true"].astype(str)
    df["hgp_str"] = df["hypergraphpercol"].astype(str)
    df["hdb_str"] = df["hdbscan"].astype(str)

    fig_true = px.scatter(
        df, x="x", y="y", color="true_str",
        title=f"{title_prefix} - Ground truth",
        opacity=0.8
    )
    fig_hgp = px.scatter(
        df, x="x", y="y", color="hgp_str",
        title=f"{title_prefix} - HypergraphPercol (relabel)",
        opacity=0.8
    )
    fig_hdb = px.scatter(
        df, x="x", y="y", color="hdb_str",
        title=f"{title_prefix} - HDBSCAN (relabel)",
        opacity=0.8
    )

    fig_true.show()
    fig_hgp.show()
    fig_hdb.show()


for d in datasets_2d:
    name = d["name"]
    X = d["X"]
    y_true = d["y_true"]
    y_hgp = labels_hgp_aligned.get(name)
    y_hdb = labels_hdb_aligned.get(name)

    print(f"\nVisualisation 2D : {name}")
    plot_2d_results(X, y_true, y_hgp, y_hdb, title_prefix=name)



Visualisation 2D : blobs_varies_2d



Visualisation 2D : moons_plus_bruit_2d



Visualisation 2D : cercles_plus_outliers_2d


In [48]:
# Cellule 7 : visualisations 3D robustes avec go.Scatter3d + iplot

def plot_3d_single(X, labels, title, max_points=3000):
    """
    Affiche un nuage 3D coloré par 'labels' avec go.Scatter3d + iplot,
    en downsamplant si nécessaire pour éviter de cramer le front-end.
    """
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels).reshape(-1)

    if X.ndim != 2 or X.shape[1] < 3:
        raise ValueError(f"X doit être de shape (n_samples, >=3), reçu {X.shape}")

    if len(X) != len(labels):
        raise ValueError(f"Taille incompatible: len(X)={len(X)}, len(labels)={len(labels)}")

    # Downsampling si trop de points
    n = len(X)
    if n > max_points:
        idx = np.random.choice(n, size=max_points, replace=False)
        X = X[idx]
        labels = labels[idx]

    unique_labels = np.unique(labels)

    fig = go.Figure()

    for lab in unique_labels:
        mask = labels == lab
        if not np.any(mask):
            continue

        lab_name = f"cluster {lab}"
        fig.add_trace(go.Scatter3d(
            x=X[mask, 0].tolist(),
            y=X[mask, 1].tolist(),
            z=X[mask, 2].tolist(),
            mode='markers',
            name=lab_name,
            marker=dict(
                size=3,
                opacity=0.85
            )
        ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
        ),
        margin=dict(l=0, r=0, b=0, t=30),
        showlegend=True
    )

    # Utilise iplot plutôt que fig.show(), pour éviter les caprices du renderer
    iplot(fig)


def plot_3d_results(X, y_true, y_hgp, y_hdb, title_prefix=""):
    X = np.asarray(X, dtype=float)
    y_true = np.asarray(y_true).reshape(-1)
    y_hgp = np.asarray(y_hgp).reshape(-1)
    y_hdb = np.asarray(y_hdb).reshape(-1)

    if not (len(X) == len(y_true) == len(y_hgp) == len(y_hdb)):
        raise ValueError(
            f"Taille incohérente : len(X)={len(X)}, len(y_true)={len(y_true)}, "
            f"len(y_hgp)={len(y_hgp)}, len(y_hdb)={len(y_hdb)}"
        )

    print(f"  → 3D Ground truth")
    plot_3d_single(X, y_true, f"{title_prefix} - Ground truth")

    print(f"  → 3D HypergraphPercol (relabel)")
    plot_3d_single(X, y_hgp, f"{title_prefix} - HypergraphPercol (relabel)")

    print(f"  → 3D HDBSCAN (relabel)")
    plot_3d_single(X, y_hdb, f"{title_prefix} - HDBSCAN (relabel)")


# Boucle sur les datasets 3D
for d in datasets_3d:
    name = d["name"]
    X = d["X"]
    y_true = d["y_true"]
    y_hgp = labels_hgp_aligned.get(name)
    y_hdb = labels_hdb_aligned.get(name)

    print(f"\nVisualisation 3D : {name}")
    plot_3d_results(X, y_true, y_hgp, y_hdb, title_prefix=name)



Visualisation 3D : blobs_anisotropes_3d
  → 3D Ground truth


  → 3D HypergraphPercol (relabel)


  → 3D HDBSCAN (relabel)



Visualisation 3D : spirales_plus_bruit_3d
  → 3D Ground truth


  → 3D HypergraphPercol (relabel)


  → 3D HDBSCAN (relabel)
